In [2]:
import re
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

def extract_medicines(text):
    medicines = []
    current = {}
    sample_date = ""
    for line in text.splitlines():
        key, sep, value = line.partition(":")
        if not sep:
            continue
        key, value = key.strip().lower(), value.strip()
        if key == "date":
            sample_date = value
            if current:
                current["startDate"] = value
            continue
        if key == "medicine":
            if current:
                medicines.append(current)
            current = {"name": value, "strength": "", "dose": "", "times": [], "food": "any", "startDate": sample_date, "endDate": ""}
        elif not current:
            continue
        elif key == "strength":
            current["strength"] = value
        elif key == "dose":
            current["dose"] = value
        elif key == "frequency":
            frequency = value.upper()
            current["times"] = {"OD": ["08:00"], "BD": ["08:00", "20:00"], "TDS": ["08:00", "14:00", "20:00"]}.get(frequency, [])
        elif key == "food":
            current["food"] = "after" if value.lower().startswith("after") else "before" if value.lower().startswith("before") else "any"
        elif key == "duration":
            match = re.search(r"(\d+)\s+days?", value, re.I)
            if match and current.get("startDate"):
                from datetime import date, timedelta
                current["endDate"] = (date.fromisoformat(current["startDate"]) + timedelta(days=int(match.group(1)) - 1)).isoformat()
    if current:
        medicines.append(current)
    return medicines

text = (PROJECT_ROOT / "data" / "prescriptions" / "prescription_01.txt").read_text(encoding="utf-8")
medicines = extract_medicines(text)
for medicine in medicines:
    print(medicine)


{'name': 'Paracetamol', 'strength': '500 mg', 'dose': '1 tablet', 'times': ['08:00', '20:00'], 'food': 'after', 'startDate': '2026-11-18', 'endDate': '2026-11-22'}
